In [ ]:

# ============================================================
# CS 340 - Project Two
# Grazioso Salvare Animal Rescue Dashboard
# Created by: Ambren Cavazos
# Dashboard ID: AAC-CS340-2026
#
# This dashboard uses MongoDB, PyMongo, Pandas, Dash,
# Plotly Express, and Dash Leaflet to display and analyze
# animal shelter records for rescue dog selection.
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

# Jupyter-compatible Dash application
from jupyter_dash import JupyterDash

# Dashboard components and callback dependencies
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

# Visualization and data processing
import plotly.express as px
import pandas as pd

# Logo encoding and file handling
import base64
import os

# Import the CRUD module created in Project One
from CRUD_Python_Module import AnimalShelter

# Configure Jupyter Dash for the Codio environment
JupyterDash.infer_jupyter_proxy_config()


# ============================================================
# 2. CONNECT TO THE MONGODB DATABASE
# ============================================================

# MongoDB account credentials
# Enter the actual password in Codio.
# Do not publish your password to GitHub.
username = "aacuser"
password = os.environ.get("AAC_MONGO_PASSWORD")


if not password:
    raise ValueError(
        "Set AAC_MONGO_PASSWORD in the Codio environment "
        "before running this notebook."
    )

# Create an instance of the AnimalShelter CRUD class
db = AnimalShelter(username, password)

# Retrieve all animal records from MongoDB
records = db.read({})

# Convert MongoDB records into a Pandas DataFrame
df = pd.DataFrame.from_records(records)

# Remove MongoDB's ObjectId column because ObjectId
# values cannot be directly serialized into Dash tables.
df.drop(columns=["_id"], inplace=True, errors="ignore")

# Display basic information for debugging
print("MongoDB connection successful.")
print("Total animal records:", len(df))
print("Available columns:", list(df.columns))


# ============================================================
# 3. CREATE THE DASH APPLICATION
# ============================================================

app = JupyterDash(__name__)


# ============================================================
# 4. LOAD THE GRAZIOSO SALVARE LOGO
# ============================================================

# Change this path if the logo is in another folder.
image_filename = "Grazioso Salvare Logo.png"

# Encode the logo so it can be displayed directly in Dash.
if os.path.isfile(image_filename):
    with open(image_filename, "rb") as image_file:
        encoded_image = base64.b64encode(
            image_file.read()
        ).decode("utf-8")

    logo_component = html.Img(
        src="data:image/png;base64,{}".format(encoded_image),
        style={
            "width": "180px",
            "height": "auto"
        }
    )

else:
    print("WARNING: Logo file not found:", image_filename)

    # Display a temporary message until the file is located.
    logo_component = html.H3("Grazioso Salvare")


# ============================================================
# 5. DASHBOARD LAYOUT
# ============================================================

app.layout = html.Div([

    # --------------------------------------------------------
    # Logo and dashboard identification
    # --------------------------------------------------------

    html.Center([

        # Clicking the logo opens the SNHU website.
        html.A(
            logo_component,
            href="https://www.snhu.edu",
            target="_blank"
        ),

        html.H1(
            "CS 340 - Grazioso Salvare Rescue Dashboard"
        ),

        html.H3(
            "Created by: Ambren Cavazos"
        ),

        html.P(
            "Dashboard ID: AAC-CS340-2026"
        )

    ]),

    html.Hr(),

    # --------------------------------------------------------
    # Rescue category filtering controls
    # --------------------------------------------------------

    html.Div([

        html.H3("Select Rescue Category"),

        # RadioItems allows one rescue category at a time.
        dcc.RadioItems(

            id="filter-type",

            options=[
                {
                    "label": "Reset (Show All Animals)",
                    "value": "reset"
                },
                {
                    "label": "Water Rescue",
                    "value": "water"
                },
                {
                    "label": "Mountain/Wilderness Rescue",
                    "value": "mountain"
                },
                {
                    "label": "Disaster/Individual Tracking",
                    "value": "disaster"
                }
            ],

            # Show all animals when the dashboard first loads.
            value="reset",

            labelStyle={
                "display": "inline-block",
                "marginRight": "20px",
                "marginBottom": "10px"
            },

            inputStyle={
                "marginRight": "6px"
            }
        )

    ], style={
        "textAlign": "center",
        "padding": "15px"
    }),

    html.Hr(),

    # --------------------------------------------------------
    # Interactive DataTable
    # --------------------------------------------------------

    dash_table.DataTable(

        id="datatable-id",

        # Create table columns using the DataFrame fields.
        columns=[
            {
                "name": column,
                "id": column,
                "deletable": False,
                "selectable": True
            }
            for column in df.columns
        ],

        # Initially display all animal records.
        data=df.to_dict("records"),

        # Permit selection of one animal at a time.
        row_selectable="single",

        # Initially select the first row if data exists.
        selected_rows=[0] if not df.empty else [],

        # Display ten records per page.
        page_size=10,

        # Enable built-in sorting and filtering.
        sort_action="native",
        filter_action="native",

        # Enable horizontal scrolling for many columns.
        style_table={
            "overflowX": "auto"
        },

        # Improve readability of table cells.
        style_cell={
            "textAlign": "left",
            "padding": "8px",
            "minWidth": "100px",
            "maxWidth": "250px",
            "overflow": "hidden",
            "textOverflow": "ellipsis"
        },

        # Format column headings.
        style_header={
            "backgroundColor": "#D2F3FF",
            "fontWeight": "bold"
        },

        # Highlight selected rows.
        style_data_conditional=[
            {
                "if": {"state": "selected"},
                "backgroundColor": "#D2F3FF",
                "border": "1px solid #0074D9"
            }
        ]
    ),

    html.Br(),
    html.Hr(),

    # --------------------------------------------------------
    # Pie chart and geolocation map
    # --------------------------------------------------------

    # Display the chart and map side by side.
    html.Div(

        style={
            "display": "flex",
            "flexWrap": "wrap",
            "gap": "20px",
            "width": "100%"
        },

        children=[

            # Breed distribution chart container
            html.Div(
                id="graph-id",
                style={
                    "flex": "1 1 420px",
                    "minWidth": "0"
                }
            ),

            # Animal geolocation map container
            html.Div(
                id="map-id",
                style={
                    "flex": "1 1 420px",
                    "minWidth": "0"
                }
            )

        ]
    )

], style={
    "padding": "20px",
    "fontFamily": "Arial, sans-serif"
})


# ============================================================
# 6. RESCUE CATEGORY FILTERING CALLBACK
# ============================================================

# This callback executes MongoDB queries through the
# AnimalShelter CRUD module whenever a user selects a filter.

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    # --------------------------------------------------------
    # WATER RESCUE
    # --------------------------------------------------------
    # Preferred breeds:
    # Labrador Retriever Mix
    # Chesapeake Bay Retriever
    # Newfoundland
    #
    # Sex: Intact Female
    # Age: 26 to 156 weeks
    # --------------------------------------------------------

    if filter_type == "water":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },

            "sex_upon_outcome": "Intact Female",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # --------------------------------------------------------
    # MOUNTAIN / WILDERNESS RESCUE
    # --------------------------------------------------------
    # Preferred breeds:
    # German Shepherd
    # Alaskan Malamute
    # Old English Sheepdog
    # Siberian Husky
    # Rottweiler
    #
    # Sex: Intact Male
    # Age: 26 to 156 weeks
    # --------------------------------------------------------

    elif filter_type == "mountain":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # --------------------------------------------------------
    # DISASTER / INDIVIDUAL TRACKING
    # --------------------------------------------------------
    # Preferred breeds:
    # Doberman Pinscher
    # German Shepherd
    # Golden Retriever
    # Bloodhound
    # Rottweiler
    #
    # Sex: Intact Male
    # Age: 20 to 300 weeks
    # --------------------------------------------------------

    elif filter_type == "disaster":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # --------------------------------------------------------
    # RESET FILTER
    # --------------------------------------------------------
    # An empty query retrieves all animal records.
    # --------------------------------------------------------

    else:
        query = {}

    # Execute the query using the Project One CRUD module.
    filtered_records = db.read(query)

    # Convert the results into a Pandas DataFrame.
    filtered_df = pd.DataFrame.from_records(
        filtered_records
    )

    # Return an empty table if there are no matching records.
    if filtered_df.empty:
        return []

    # Remove the MongoDB ObjectId field.
    filtered_df.drop(
        columns=["_id"],
        inplace=True,
        errors="ignore"
    )

    # Return records in the format expected by Dash.
    return filtered_df.to_dict("records")




# ============================================================
# 7. BREED DISTRIBUTION PIE CHART CALLBACK
# ============================================================

# Update the pie chart whenever the DataTable changes.
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):

    # Get the currently displayed animal records.
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)

    # Handle filters that return no records.
    if dff.empty:
        return html.P(
            "No animal records available for the chart."
        )

    # Count animals belonging to each breed.
    breed_counts = (
        dff["breed"]
        .fillna("Unknown")
        .value_counts()
    )

    # Keep only the 10 most common breeds.
    top_breeds = breed_counts.head(10)

    # Combine all remaining breeds into one category.
    other_count = breed_counts.iloc[10:].sum()

    # Add the Other Breeds category if needed.
    if other_count > 0:
        top_breeds = pd.concat([
            top_breeds,
            pd.Series({"Other Breeds": other_count})
        ])

    # Convert breed counts into a DataFrame.
    chart_df = top_breeds.reset_index()
    chart_df.columns = ["breed", "count"]

    # Create the pie chart.
    fig = px.pie(
        chart_df,
        names="breed",
        values="count",
        title="Animal Breed Distribution",
        hole=0.35
    )

    # Display percentages inside the slices.
    fig.update_traces(
        textposition="inside",
        textinfo="percent",
        hovertemplate=(
            "%{label}<br>"
            "Animals: %{value}<br>"
            "Percentage: %{percent}<extra></extra>"
        )
    )

    # Improve layout and legend readability.
    fig.update_layout(
        height=500,
        margin=dict(l=10, r=10, t=60, b=20),
        legend_title_text="Breed",
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.1,
            xanchor="center",
            x=0.5
        )
    )

    # Return the chart to the dashboard.
    return [
        dcc.Graph(
            figure=fig,
            style={"width": "100%"}
        )
    ]

# ============================================================
# 8. SELECTED COLUMN HIGHLIGHTING CALLBACK
# ============================================================

# Highlight columns selected by the user in the DataTable.

@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):

    # Dash may return None when no columns are selected.
    if selected_columns is None:
        selected_columns = []

    return [
        {
            "if": {
                "column_id": column
            },
            "backgroundColor": "#D2F3FF"
        }
        for column in selected_columns
    ]


# ============================================================
# 9. GEOLOCATION MAP CALLBACK
# ============================================================

# This callback updates the map when the user selects
# an animal or when the visible table records change.
#
# It uses column names rather than numeric positions,
# making the code easier to understand and maintain.

@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):

    # Obtain the current table records.
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)

    # Handle filters that return no animals.
    if dff.empty:
        return html.P(
            "No animal records available to map."
        )

    # Default to the first animal if no row is selected.
    if not index:
        row = 0
    else:
        row = index[0]

    # Prevent invalid row indexes after filtering.
    if row < 0 or row >= len(dff):
        row = 0

    # Retrieve the selected animal's coordinates.
    latitude = pd.to_numeric(
        dff.iloc[row]["location_lat"],
        errors="coerce"
    )

    longitude = pd.to_numeric(
        dff.iloc[row]["location_long"],
        errors="coerce"
    )

    # Handle missing or invalid coordinates.
    if pd.isna(latitude) or pd.isna(longitude):
        return html.P(
            "Location data is unavailable for this animal."
        )

    # Retrieve the selected animal's name and breed.
    animal_name = str(
        dff.iloc[row]["name"]
    )

    animal_breed = str(
        dff.iloc[row]["breed"]
    )

    # Create the map centered on the selected animal.
    return [

        dl.Map(

            style={
                "width": "100%",
                "height": "500px"
            },

            center=[
                float(latitude),
                float(longitude)
            ],

            zoom=10,

            children=[

                # Display the base map.
                dl.TileLayer(),

                # Display the selected animal's location.
                dl.Marker(

                    position=[
                        float(latitude),
                        float(longitude)
                    ],

                    children=[

                        # Show the animal's breed on hover.
                        dl.Tooltip(
                            animal_breed
                        ),

                        # Show the animal's name when clicked.
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(animal_name),
                            html.H4("Breed"),
                            html.P(animal_breed)
                        ])

                    ]
                )

            ]
        )

    ]


# ============================================================
# 10. RUN THE DASHBOARD
# ============================================================

# Use external mode because it worked successfully
# in the Codio environment during Module Six.
#
# Port 8051 avoids potential conflicts with an
# application already using the default port 8050.

app.run_server(
    mode="external",
    port=8051,
    debug=False
)
